## Topic: Sequential Chain in LangChain

### Agenda
- 1. Introduction of Sequential Chain

- 2. Architecture of Sequential Chain

- 3. Types of Sequential Chains

- 4. Real-World Example

- 5. Limitations of Sequential Chains

### 1. Introduction of Sequential Chain

- Definition:
    - A Sequential Chain is a type of LangChain chain that connects multiple sub-chains in a fixed, linear order. Each sub-chain runs one after another, and the output of one becomes the input to the next.


In [ ]:
"""   - Key Rules of SimpleSequentialChain:

┌─────────────────────────────────────────────────────┐
│         SimpleSequentialChain RULES                 │
├─────────────────────────────────────────────────────┤
│ 1. Each sub-chain MUST have exactly ONE input var   │
├─────────────────────────────────────────────────────┤
│ 2. Each sub-chain MUST have exactly ONE output var  │
├─────────────────────────────────────────────────────┤
│ 3. Output of Chain N → Input of Chain N+1           │
├─────────────────────────────────────────────────────┤
│ 4. Variable names DON'T need to match               │
│    (Chain 1 output "text" → Chain 2 input "idea")   │
├─────────────────────────────────────────────────────┤
│ 5. First chain's input = overall chain's input      │
├─────────────────────────────────────────────────────┤
│ 6. Last chain's output = overall chain's output     │
└─────────────────────────────────────────────────────┘

"""

### 2. Architecture of Sequential Chain

In [ ]:
""" 
                 SEQUENTIAL CHAIN

                     INPUT
                       │
                       ▼
                ┌─────────────┐
                │    STEP 1   │
                │ Prompt      │
                │ Model       │
                │ Parser      │
                └──────┬──────┘
                       │
                       ▼
                    OUTPUT 1
                       │
                       ▼
                ┌─────────────┐
                │    STEP 2   │
                │ Prompt      │
                │ Model       │
                │ Parser      │
                └──────┬──────┘
                       │
                       ▼
                    OUTPUT 2
                       │
                       ▼
                ┌─────────────┐
                │    STEP 3   │
                │ Prompt      │
                │ Model       │
                │ Parser      │
                └──────┬──────┘
                       │
                       ▼
                  FINAL OUTPUT

"""

### 3. Types of Sequential Chains


In [ ]:
"""  - Two Types of Sequential Chains

┌──────────────────────────────────────────────────────────────┐
│              SEQUENTIAL CHAIN TYPES                           │
│                                                              │
│  ┌────────────────────────────────────────────────────────┐  │
│  │  1. SimpleSequentialChain                              │  │
│  │     ├── Single input variable                          │  │
│  │     ├── Single output variable                         │  │
│  │     ├── Each sub-chain has ONE input and ONE output    │  │
│  │     └── Easiest to use, most common                    │  │
│  └────────────────────────────────────────────────────────┘  │
│                                                              │
│  ┌────────────────────────────────────────────────────────┐  │
│  │  2. SequentialChain (Full)                             │  │
│  │     ├── Multiple input variables                       │  │
│  │     ├── Multiple output variables                      │  │
│  │     ├── Each sub-chain can have MULTIPLE inputs/outputs│  │
│  │     ├── Explicit variable name mapping                 │  │
│  │     └── More flexible, more complex                    │  │
│  └────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘


"""

### 4. Real-World Example

#### 4.1: SimpleSequentialChain Example

- Idea:
    - step1: prompt1 (topic)
        - prompt1: Generate a detailed report based on {topic}

    - step2: LLm (input: perform based on prompt1 topic)
        - input: prompt1
        - response1 : Generate a Detailed report base on {topic}

    - step3: LLm (input: response of the step2) 
        - prompt2: Give a summary a response1
        - input: prompt2
        - response2: generate summary

    - step4: Final output
        - response : give a summary on {topic}

In [ ]:
## 4.1: SimpleSequentialChain Example 1

# import necessary libraries
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# define the model
model = ChatOpenAI()

# Prompt 1:
prompt1 = PromptTemplate(
    template="Generate a Detailed report based on:  {topic}",
    input_variables=["topic"]
)

# Prompt2
prompt2 = PromptTemplate(
    template= "Generate a 5 pointer summary form the following text \n {text}",
    input_variables=["text"]
)

# parser
parser = StrOutputParser()

# Define the chain
chain = prompt1 | model | parser | prompt2 | model | parser

# response
response = chain.invoke(
    {
        "topic": "Generative AI"
    }
)

print(response)

# for Visualization of chain
chain.get_graph().print_ascii()


#### 4.1: SimpleSequentialChain Example 2
- Idea:
    - step1: prompt1 (genre)
        - prompt1: creative one-line movie idea for a {genre} film

    - step2: LLm (input: perform based on prompt1 topic)
        - input: prompt1
        - response1 :  one-line movie idea for a {genre} film

    - step3: LLm (input: response of the step2) 
        - prompt2: Write a 3-sentence movie synopsis based on this idea {idea}
        - input: prompt2
        - response2:  3-sentence movie synopsis

    - step4: Final output
        - response : give a summary on {topic}

In [ ]:
from langchain.chains import SimpleSequentialChain, LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# ─── Sub-Chain 1: Generate a movie idea ───
idea_template = PromptTemplate(
    input_variables=["genre"],
    template="Come up with a creative one-line movie idea for a {genre} film."
)
idea_chain = LLMChain(llm=llm, prompt=idea_template)

# ─── Sub-Chain 2: Write a synopsis from the idea ───
synopsis_template = PromptTemplate(
    input_variables=["idea"],
    template="Write a 3-sentence movie synopsis based on this idea:\n{idea}"
)
synopsis_chain = LLMChain(llm=llm, prompt=synopsis_template)

# ─── Sub-Chain 3: Write a review from the synopsis ───
review_template = PromptTemplate(
    input_variables=["synopsis"],
    template="Write a short, enthusiastic movie review based on this synopsis:\n{synopsis}"
)
review_chain = LLMChain(llm=llm, prompt=review_template)

# ─── Combine into a SimpleSequentialChain ───
overall_chain = SimpleSequentialChain(
    chains=[idea_chain, synopsis_chain, review_chain],
    verbose=True  # Shows each step's input/output
)

# ─── Run it! ───
result = overall_chain.run("sci-fi thriller")

print("\n" + "="*60)
print("FINAL OUTPUT:")
print("="*60)
print(result)

In [ ]:
### - What Happens Step by Step (verbose=True output):
""" 
> Entering new SimpleSequentialChain chain...

[Chain 1 Input]:  sci-fi thriller
[Chain 1 Output]: "A time-traveling detective must solve her own murder 
                    before it happens in a dystopian 2147."

[Chain 2 Input]:  "A time-traveling detective must solve her own murder 
                    before it happens in a dystopian 2147."
[Chain 2 Output]: "In 2147, Detective Mara Voss discovers a dead body 
                    that is unmistakably her own. Using illegal time-jump 
                    technology, she races back 72 hours to prevent her 
                    assassination. But every change she makes creates a 
                    more dangerous timeline."

[Chain 3 Input]:  "In 2147, Detective Mara Voss discovers..."
[Chain 3 Output]: "★★★★★ A mind-bending masterpiece! The concept of a 
                    detective investigating her own future murder is 
                    brilliantly executed. The time-loop mechanics keep 
                    you guessing until the very last frame. A must-watch 
                    for any sci-fi fan!"

> Finished chain.



"""

### 4.2: SequentialChain (Full) Example

In [ ]:
# ### 4.2: SequentialChain (Full) Example
from langchain.chains import SequentialChain, LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Define LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# ─── Sub-Chain 1: Generate outline and tone ───
outline_template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="""Create a blog outline and determine the writing tone.

Topic: {topic}
Target Audience: {audience}

Respond in this exact format:
OUTLINE:
1. [Section 1]
2. [Section 2]
3. [Section 3]

TONE: [formal/casual/technical/conversational]"""
)


outline_chain = LLMChain(
    llm=llm,
    prompt=outline_template,
    output_key="outline_and_tone"  # ← Named output!
)

# ─── Sub-Chain 2: Write the draft ───
draft_template = PromptTemplate(
    input_variables=["outline_and_tone", "word_count"],
    template="""Write a blog post following this outline and tone:

{outline_and_tone}

Target word count: {word_count} words.
Write the complete blog post now."""
)
draft_chain = LLMChain(
    llm=llm,
    prompt=draft_template,
    output_key="draft"  # ← Named output!
)

# ─── Sub-Chain 3: Generate SEO metadata ───
seo_template = PromptTemplate(
    input_variables=["draft", "topic"],
    template="""Based on this blog post about "{topic}", generate SEO metadata:

Blog Post:
{draft}

Respond in this format:
TITLE: [SEO-optimized title, max 60 chars]
META: [Meta description, max 155 chars]
KEYWORDS: [5 comma-separated keywords]"""
)
seo_chain = LLMChain(
    llm=llm,
    prompt=seo_template,
    output_key="seo_metadata"  # ← Named output!
)

# ─── Combine into a Full SequentialChain ───
overall_chain = SequentialChain(
    chains=[outline_chain, draft_chain, seo_chain],
    input_variables=["topic", "audience", "word_count"],    # ← ALL inputs
    output_variables=["outline_and_tone", "draft", "seo_metadata"],  # ← ALL outputs
    verbose=True
)

# ─── Run it! ───
result = overall_chain({
    "topic": "Artificial Intelligence in Healthcare",
    "audience": "Medical professionals",
    "word_count": "300"
})

# Access ALL intermediate and final outputs!
print(" OUTLINE & TONE:")
print(result["outline_and_tone"])

print("\n DRAFT:")
print(result["draft"])

print("\n SEO METADATA:")
print(result["seo_metadata"])

### 5. Limitations of Sequential Chains

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│          SEQUENTIAL CHAIN LIMITATIONS                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ 1. FIXED ORDER ONLY                                         │
│    Can't skip steps, loop back, or branch conditionally.    │
│    Step 1 → Step 2 → Step 3. Always. No exceptions.         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ 2. NO PARALLEL EXECUTION                                    │
│    Steps run one after another. If Step 1 takes 5s and      │
│    Step 2 takes 5s, total = 10s. Can't run in parallel.     │
├─────────────────────────────────────────────────────────────┤ 
│                                                             │
│ 3. NO STREAMING (Legacy)                                    │
│    Legacy SequentialChain doesn't support .stream().        │
│    You wait for the ENTIRE pipeline to finish.              │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│ 4. ERROR PROPAGATION                                        │
│    If Step 2 fails, the entire chain crashes.               │
│    No built-in retry or fallback per step.                  │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│ 5. VERBOSE BOILERPLATE                                      │
│    Lots of code for simple pipelines.                       │
│    LLMChain + PromptTemplate + output_key + input_variables │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│ 6. DEPRECATED                                               │
│    LangChain team recommends LCEL for all new projects.     │
│    SequentialChain may be removed in future versions.       │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│ 7. NO DYNAMIC ROUTING                                       │
│    Can't choose different chains based on intermediate      │
│    results. (Use RunnableBranch in LCEL instead.)           │
└─────────────────────────────────────────────────────────────┘

"""